In [25]:
'''
Data cleaning

1. Negative quantities (refunds)
I will retain negative quantities and create a mechanism to distinguish refunds from purchases,
enabling both net revenue calculation and refund analysis.
2. Missing customer IDs
Transactions with missing CustomerIDs will be included in revenue analysis but excluded
from customer-level analysis such as retention and segmentation, as they cannot be reliably tracked.
3. Zero price transactions
Transactions with zero price will be excluded as they
do not contribute to revenue or customer behavior insights relevant to the project’s objectives

'''

'\nData cleaning\n\n1. Negative quantities (refunds)\nI will retain negative quantities and create a mechanism to distinguish refunds from purchases,\nenabling both net revenue calculation and refund analysis.\n2. Missing customer IDs\nTransactions with missing CustomerIDs will be included in revenue analysis but excluded\nfrom customer-level analysis such as retention and segmentation, as they cannot be reliably tracked.\n3. Zero price transactions\nTransactions with zero price will be excluded as they\ndo not contribute to revenue or customer behavior insights relevant to the project’s objectives\n\n'

In [26]:
import pandas as pd
import numpy as np
import sqlite3

df = pd.read_csv('online_retail_II.csv')
df = df.rename(columns={
    'Invoice': 'invoice_id',
    'StockCode': 'product_id',
    'Description': 'product_desc',
    'Quantity': 'quantity',
    'Price': 'price',
    'Customer ID': 'customer_id',
    'Country': 'country',
    'InvoiceDate': 'invoice_date'
})

df = df[df['price'] != 0]

df['revenue'] = df['quantity'] * df['price']
df['is_refund']=np.where(df['quantity'] < 0, True, False)
df['invoice_date'] = pd.to_datetime(df['invoice_date'])

df['product_desc'] = df['product_desc'].str.strip().str.lower()

customers_df=df[['customer_id', 'country']].dropna(subset=['customer_id']).drop_duplicates(subset=['customer_id'])
product_df = (df[['product_id', 'product_desc']].dropna(subset=['product_id']).drop_duplicates(subset=['product_id']))
orders_df = (df[['invoice_id', 'invoice_date', 'customer_id']].drop_duplicates(subset=['invoice_id']))
order_items_df = df[['invoice_id', 'product_id', 'price', 'quantity', 'revenue', 'is_refund']]




In [27]:
conn = sqlite3.connect("retail.db")
cursor = conn.cursor()
# Enable foreign keys
cursor.execute("PRAGMA foreign_keys = ON;")

#  DROP TABLES (clean reset)
cursor.executescript("""
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
""")

cursor.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    country TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE products (
    product_id TEXT PRIMARY KEY,
    product_desc TEXT
);
""")

cursor.execute("""
CREATE TABLE orders (
    invoice_id TEXT PRIMARY KEY,
    invoice_date TEXT,
    customer_id INTEGER,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS order_items (
    order_item_id INTEGER PRIMARY KEY,
    invoice_id TEXT,
    product_id TEXT,
    price REAL,
    quantity INTEGER,
    revenue REAL,
    is_refund BOOLEAN,
    FOREIGN KEY (invoice_id) REFERENCES orders(invoice_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

conn.commit()

# Populate tables with data from dataframes
customers_df.to_sql('customers', conn, if_exists='append', index=False)
product_df.to_sql('products', conn, if_exists='append', index=False)
orders_df.to_sql('orders', conn, if_exists='append', index=False)
order_items_df.to_sql('order_items', conn, if_exists='append', index=False)
conn.commit()



# Query: top 10 products by NET revenue
query = """
SELECT order_items.product_id, product_desc, SUM(revenue) as total_revenue, SUM(quantity) as units_sold from order_items
INNER JOIN products ON order_items.product_id =products.product_id
GROUP BY order_items.product_id, product_desc
ORDER BY total_revenue DESC
LIMIT 10
"""
product_df = pd.read_sql_query(query, conn)
product_df.to_csv("product_revenue.csv", index=False)

# Query: Revenue in each month
query1 = """
SELECT strftime('%m', invoice_date) as month, SUM(revenue) as total_revenue
from order_items INNER JOIN orders
ON order_items.invoice_id = orders.invoice_id
GROUP BY month
ORDER BY total_revenue DESC
"""
monthly_df = pd.read_sql_query(query1, conn)

monthly_df.to_csv("monthly_revenue.csv", index=False)


# Query: Products with highest refund rate
query2 = """
SELECT
    oi.product_id,
    product_desc,
    oi.total_sold_quantity,
    oi.total_refunded_quantity,
    ROUND(oi.total_refunded_quantity * 100.0 / NULLIF(oi.total_sold_quantity, 0), 2) AS refund_rate
FROM (
    SELECT
        SUM(CASE WHEN quantity > 0 THEN quantity ELSE 0 END) AS total_sold_quantity,
        SUM(abs(CASE WHEN quantity < 0 THEN quantity ELSE 0 END)) AS total_refunded_quantity,
        product_id
    FROM order_items
    GROUP BY product_id
) AS oi
INNER JOIN products ON oi.product_id = products.product_id
WHERE oi.total_sold_quantity >= 50
ORDER BY refund_rate DESC
LIMIT 15
"""

refund_df = pd.read_sql_query(query2, conn)

refund_df.to_csv("refund_analysis.csv", index=False)

# Query: Customer segmentation, high value and low value customers
query3 = """
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(revenue) AS total_revenue
    FROM order_items
    INNER JOIN orders
        ON order_items.invoice_id = orders.invoice_id
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
)

SELECT
    customer_id,
    total_revenue,

    NTILE(5) OVER (
        ORDER BY total_revenue DESC
    ) AS customer_segment

FROM customer_revenue

ORDER BY total_revenue DESC;
"""

# Query: Customer segments' revenue contribution, AOV, AvPurchaseFreq, RefundedRevRate, AvgUniqueProducts
query4 = """
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(revenue) AS total_revenue,
        COUNT(DISTINCT orders.invoice_id) AS total_orders,
        SUM(abs(CASE WHEN revenue < 0 THEN revenue ELSE 0 END)) AS refunded_revenue,
        SUM(CASE WHEN revenue > 0 THEN revenue ELSE 0 END) AS gross_revenue,
        COUNT(DISTINCT product_id) AS unique_products
    FROM order_items
    INNER JOIN orders
        ON order_items.invoice_id = orders.invoice_id
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
),

customer_segments AS (
  SELECT
    customer_id,
    total_revenue,
    total_orders,
    refunded_revenue,
    gross_revenue,
    unique_products,
    NTILE(5) OVER (
        ORDER BY total_revenue DESC
    ) AS customer_segment

FROM customer_revenue
)

SELECT customer_segment, SUM(total_revenue) AS segment_revenue,

    ROUND(
        100.0 * SUM(total_revenue)
        / SUM(SUM(total_revenue)) OVER (),
        2
    ) AS revenue_pct,

    ROUND(
        SUM(total_revenue) * 1.0
        / SUM(total_orders),
        2
    ) AS avg_order_value,

    ROUND(
        SUM(total_orders) * 1.0
        / COUNT(customer_id),
        2
    ) AS avg_purchase_frequency,

    ROUND(SUM(refunded_revenue) * 100.0 / NULLIF(SUM(gross_revenue), 0), 2)
    AS refunded_revenue_rate,

    ROUND(AVG(unique_products), 2) AS avg_unique_products

FROM customer_segments
GROUP BY customer_segment
ORDER BY customer_segment
"""

customer_segment_df = pd.read_sql_query(query4, conn)

customer_segment_df.to_csv("customers_segments.csv", index=False)

# Query: Product segmentation, high value and low value products
query5 = """
WITH product_revenue AS (
    SELECT order_items.product_id, product_desc, SUM(revenue) as total_revenue, SUM(quantity) as units_sold from order_items
    INNER JOIN products ON order_items.product_id =products.product_id
    GROUP BY order_items.product_id, product_desc
),

product_segments AS (
    SELECT
        total_revenue,
        NTILE(5) OVER (ORDER BY total_revenue DESC) AS product_segment
    FROM product_revenue
)

SELECT
    product_segment,
    SUM(total_revenue) AS segment_revenue,
    ROUND(100.0 * SUM(total_revenue) / SUM(SUM(total_revenue)) OVER (), 2) AS revenue_pct
FROM product_segments
GROUP BY product_segment
ORDER BY product_segment;
"""

segment_df = pd.read_sql_query(query5, conn)

segment_df.to_csv("product_segments.csv", index=False)

# Query: Recurring and one-off customers and their impact on revenue
query6 = """
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(revenue) AS total_revenue,
        COUNT(DISTINCT orders.invoice_id) AS invoice_count
    FROM order_items
    INNER JOIN orders
        ON order_items.invoice_id = orders.invoice_id
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
),

customer_type AS (
  SELECT
    customer_id,
    total_revenue,
    CASE WHEN invoice_count > 1 THEN 'recurring' ELSE 'one-off' END AS type

FROM customer_revenue

)

SELECT type AS customer_type,
COUNT(customer_id) as num_customers,
ROUND(100.0 * COUNT(customer_id) / (SELECT COUNT(*) FROM customer_type), 2) AS pct_customers,
SUM(total_revenue) AS total_revenue,
ROUND(100.0 * SUM(total_revenue) / (SELECT SUM(total_revenue) FROM customer_type), 2) AS pct_revenue

FROM customer_type
GROUP BY type
"""
retention_df = pd.read_sql_query(query6, conn)

retention_df.to_csv("retention_analysis.csv", index=False)

# Can test different queries here
df_result = pd.read_sql_query(query, conn)
print(df_result)

conn.close()


  product_id                        product_desc  total_revenue  units_sold
0        DOT                      dotcom postage       18574.58          49
1     85123A  white hanging heart t-light holder       17927.50        6655
2      22086      paper chain kit 50's christmas       10169.36        3362
3    15056BL             edwardian parasol black        8731.25        2198
4      22111        scottie dog hot water bottle        8145.31        1588
5      84879       assorted colour bird ornament        7623.93        4642
6     85099B          jumbo bag red white spotty        7614.25        4316
7      22114   hot water bottle tea and sympathy        6523.90        1592
8     79323W                 white cherry lights        6409.72        1074
9      20679               edwardian parasol red        6205.20        1683
